In [ ]:
import polars as pl
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from datetime import datetime, timezone, timedelta, date

In [ ]:
liq_snap_path    = Path('data/BTCUSD_PERP-liquidationSnapshot')
book_ticker_path = Path('data/BTCUSD_PERP-bookTicker')
agg_trades_path  = Path('data/BTCUSD_PERP-aggTrades')
date_of_interest1 = '2023-06-25'

start_date = date(2023, 6, 25)
end_date   = date(2023, 7, 25)

liq_snap_cols = [
    "time_datetime",
    "side",
    "original_quantity", 
    "accumulated_fill_quantity",
    "price",    
    "average_price",
    "order_status",
]

agg_trades_cols = [
    'transact_time_datetime',
    'quantity', 
    'price', 
    'is_buyer_maker'
]

In [ ]:
df_liq_snap1 = (
    pl.scan_parquet(liq_snap_path / "*.parquet")
    .select(liq_snap_cols)
    .filter(
        (pl.col("time_datetime") >= datetime(start_date.year, start_date.month, start_date.day, tzinfo=timezone.utc)) &
        (pl.col("time_datetime") <  datetime(end_date.year,   end_date.month,   end_date.day,   tzinfo=timezone.utc) + timedelta(days=1))
    )
    .unique(subset=["time_datetime"])
    .sort("time_datetime")
    .collect()
)

In [ ]:
df_liq_snap1.shape

In [ ]:
df_liq_snap1.head(5)

In [ ]:
df_agg_trades1 = (
    pl.scan_parquet(agg_trades_path / "*.parquet")
    .select(agg_trades_cols)
    .filter(
        (pl.col("transact_time_datetime") >= datetime(start_date.year, start_date.month, start_date.day, tzinfo=timezone.utc)) &
        (pl.col("transact_time_datetime") <  datetime(end_date.year,   end_date.month,   end_date.day,   tzinfo=timezone.utc) + timedelta(days=1))
    )
    .sort("transact_time_datetime")
    .collect()
)

In [ ]:
df_agg_trades1.head(5)

In [ ]:
df_book_ticker1 = (
    pl.scan_parquet(book_ticker_path / "*.parquet")
    .filter(
        (pl.col("event_time_datetime") >= datetime(start_date.year, start_date.month, start_date.day, tzinfo=timezone.utc)) &
        (pl.col("event_time_datetime") <  datetime(end_date.year,   end_date.month,   end_date.day,   tzinfo=timezone.utc) + timedelta(days=1))
    )
    .sort("event_time_datetime")
    # note that .collect() is not called
)

In [ ]:
df_book_ticker1.limit(5).collect()

# Sample experiment

In this section we go over the experimental procedure over a small sample, then
run the experiment for the entire dataset.

## Steps

### 1. Define the event set
Liquidation events of interest are those where the liquidation quantity plus 
aggTrade volumes around the liquidation event that are in the same direction as 
the liquidation event exceed the Xth percentile of the trailing Y-day 
distribution.
   - Options to consider for the event window are:
     - [t - 2s,  t + 2s]
     - **[t - 5s,  t + 5s]**
     - [t - 10s, t + 10s]
   - These are events where we have both a liquidation event *and* elevated 
     same-direction taker flow, which constitutes exactly the events we're 
     interested in for this study.

### 1.1 Exploratory analysis

### 2. Compute returns for qualifying events for multiple time windows
$$ r_{i, [a,b]} = log(P_{t_i + b}) - log(P_{t_i + a}) $$
   - Potential windows:
     - [0, 10s]
     - [5s, 30s]
     - [0, 2min]
     - [0, 5min]
     - [30s, 10min]

### 3. Sign-correct by direction
Multiply each return by the sign opposite to the liquidation direction 
(so long-liquidations get +1 because to liquidate them Binance's system 
autosells, and short-liquidations get -1 because Binance autobuys to close the 
position). After sign-correction,
positive values in the **impact window** mean "price moved with the liquidation"
and positive values in the **reversion window** mean "price reversed against the 
liquidation."

Another way to state it, which is useful for the report later:
>"We sign-correct returns such that positive values indicate price movement in 
>the direction of the cascade pressure (downward for long-liquidation events, 
>upward for short-liquidation events). Under this convention, positive returns 
>in the impact window [0, 30s] are evidence of mechanical price impact; negative 
>returns in subsequent windows are evidence of reversion."

### 4. Test statistical significance

Perform one-sample t-test for each window choice to determine if the mean of
returns is statistically different from zero. Use **Newey-West** errors, because
they correct for heteroskedasticity and auto-correlation. Newey-West not 
required when samples are i.i.d., which for this study they certainly aren't.

The above 4 steps constitute the minimum viable research project.

# Step 1 - event set

In [ ]:
def agg_trade_qty_window(liq_df, trades_df, seconds_before=5, seconds_after=5):
    trades_dt = trades_df["transact_time_datetime"]
    qty_sums = []
    for time_liq, side_liq in zip(liq_df["time_datetime"].to_list(), liq_df["side"].to_list()):
        t0 = time_liq - timedelta(seconds=seconds_before)
        t1 = time_liq + timedelta(seconds=seconds_after)
        buyer_maker = side_liq == "SELL"  # BUY liq → taker buys → is_buyer_maker=False; SELL liq → taker sells → is_buyer_maker=True
        filtered = trades_df.filter(
            (trades_dt >= t0) &
            (trades_dt <= t1) &
            (pl.col("is_buyer_maker") == buyer_maker)
        )
        qty_sums.append(filtered["quantity"].sum() if filtered.height > 0 else 0.0)
    return qty_sums

df_liq_snap1 = df_liq_snap1.with_columns(
    pl.Series("agg_qty_5s_before_5s_after", agg_trade_qty_window(df_liq_snap1, df_agg_trades1))
)
df_liq_snap1

In [ ]:
import numpy as np

times_ns  = df_liq_snap1["time_datetime"].cast(pl.Int64).to_numpy()
qtys      = df_liq_snap1["agg_qty_5s_before_5s_after"].to_numpy()
window_ns = int(timedelta(days=7).total_seconds() * 1e9)

pctiles = []
for i in range(len(times_ns)):
    t, q  = times_ns[i], qtys[i]
    mask  = (times_ns >= t - window_ns) & (times_ns <= t)
    vals  = qtys[mask]
    pctiles.append(float((vals <= q).sum() / len(vals)))

df_liq_snap1 = df_liq_snap1.with_columns(
    pl.Series("agg_qty_pctile_7d_trailing", pctiles)
)
df_liq_snap1

## Exploratory Analysis

In [ ]:
df_liq_snap1['agg_qty_5s_before_5s_after'].describe()

In [ ]:
df_liq_snap1.filter(pl.col("agg_qty_5s_before_5s_after") == pl.col("agg_qty_5s_before_5s_after").max())

In [ ]:
fig = go.Figure(go.Histogram(
    x=df_liq_snap1["agg_qty_5s_before_5s_after"].to_list(),
    nbinsx=50,
    marker_color="steelblue",
    opacity=0.8,
))
fig.update_layout(
    title=f"Distribution of agg trade qty in ±5s window around liquidation events between {start_date} and {end_date}",
    xaxis_title="agg_qty_5s_before_5s_after",
    yaxis_title="Count",
    template="plotly_white",
)
fig.show()

In [ ]:
sell_qty = df_liq_snap1.filter(pl.col("side") == "SELL")["agg_qty_5s_before_5s_after"].to_list()
buy_qty  = df_liq_snap1.filter(pl.col("side") == "BUY")["agg_qty_5s_before_5s_after"].to_list()

fig = go.Figure()
fig.add_trace(go.Histogram(x=sell_qty, nbinsx=50, name="SELL", marker_color="tomato",    opacity=0.6))
fig.add_trace(go.Histogram(x=buy_qty,  nbinsx=50, name="BUY",  marker_color="steelblue", opacity=0.6))
fig.update_layout(
    barmode="overlay",
    title=f"Distribution of agg trade qty in ±5s window by liquidation side ({start_date} to {end_date})",
    xaxis_title="agg_qty_5s_before_5s_after",
    yaxis_title="Count",
    template="plotly_white",
)
fig.show()

In [ ]:
df_liq_snap1.filter(pl.col("agg_qty_5s_before_5s_after") > 100_000)

In [ ]:
threshold = df_liq_snap1["agg_qty_5s_before_5s_after"].quantile(0.95)
df_liq_snap1.filter(
    (pl.col("side") == "BUY") &
    (pl.col("agg_qty_5s_before_5s_after") > threshold)
)

In [ ]:
threshold = df_liq_snap1["agg_qty_5s_before_5s_after"].quantile(0.95)
df_liq_snap1.filter(
    (pl.col("side") == "SELL") &
    (pl.col("agg_qty_5s_before_5s_after") > threshold)
)

In [ ]:
len(df_liq_snap1.filter(pl.col("agg_qty_5s_before_5s_after") > 100_000)) / len(df_liq_snap1)

In [ ]:
df_liq_snap1["agg_qty_5s_before_5s_after"].quantile(0.95)

In [ ]:
fig = go.Figure(go.Histogram(
    x=np.log(df_liq_snap1["agg_qty_5s_before_5s_after"]).to_list(),
    nbinsx=50,
    marker_color="steelblue",
    opacity=0.8,
))
fig.update_layout(
    title="Distribution of log agg trade qty in ±5s window around liquidation events",
    xaxis_title="log agg_qty_5s_before_5s_after",
    yaxis_title="Count",
    template="plotly_white",
)
fig.show()

In [ ]:
df_liq_per_day = (
    df_liq_snap1
    .with_columns(pl.col("time_datetime").dt.date().alias("date"))
    .group_by("date")
    .agg(pl.len().alias("count"))
    .sort("date")
)

fig = go.Figure(go.Bar(
    x=df_liq_per_day["date"].to_list(),
    y=df_liq_per_day["count"].to_list(),
    marker_color="steelblue",
    opacity=0.85,
))
fig.update_layout(
    title=f"Liquidation snapshot events per day ({start_date} to {end_date})",
    xaxis_title="Date",
    yaxis_title="Number of events",
    template="plotly_white",
)
fig.show()
df_liq_per_day

In [ ]:
df_liq_per_day = (
    df_liq_snap1
    .filter(pl.col("agg_qty_pctile_7d_trailing") >= 0.95)
    .with_columns(pl.col("time_datetime").dt.date().alias("date"))
    .group_by("date")
    .agg(pl.len().alias("count"))
    .sort("date")
)

fig = go.Figure(go.Bar(
    x=df_liq_per_day["date"].to_list(),
    y=df_liq_per_day["count"].to_list(),
    marker_color="steelblue",
    opacity=0.85,
))
fig.update_layout(
    title=f"Liquidation events per day with agg_qty_pctile_7d_trailing ≥ 0.95 ({start_date} to {end_date})",
    xaxis_title="Date",
    yaxis_title="Number of events",
    template="plotly_white",
)
fig.show()
df_liq_per_day

In [ ]:
# plot spread size in basis points; VWAP
# 
# only look around a few >= 0.95 events
df_agg_trades1.head(50)

In [ ]:
# plot the distribution of time differences between consecutive events using transact_time_datetime
diff_ms = (
    df_agg_trades1["transact_time_datetime"]
    .diff()
    .drop_nulls()
    .dt.total_milliseconds()
    .to_numpy()
)

log_diff_ms = np.log(diff_ms)

fig = go.Figure(go.Histogram(
    x=log_diff_ms,
    nbinsx=100,
    marker_color="steelblue",
    opacity=0.8,
))
fig.update_layout(
    title="Distribution of log time differences between consecutive aggTrade events",
    xaxis_title="Log time difference (ms)",
    yaxis_title="Count",
    template="plotly_white",
)
fig.show()

print(f"median: {np.median(diff_ms):.1f} ms")
print(f"p95:    {np.percentile(diff_ms, 95):.1f} ms")
print(f"max:    {diff_ms.max():.1f} ms")

In [ ]:
times_ns       = df_agg_trades1["transact_time_datetime"].cast(pl.Int64).to_numpy()
prices         = df_agg_trades1["price"].to_numpy()
qtys           = df_agg_trades1["quantity"].to_numpy()
is_buyer_maker = df_agg_trades1["is_buyer_maker"].to_numpy()

pv = prices * qtys

# bid: taker sells (is_buyer_maker=True)
# ask: taker buys  (is_buyer_maker=False)
cum_pv_bid  = np.cumsum(np.where( is_buyer_maker, pv,   0.0))
cum_qty_bid = np.cumsum(np.where( is_buyer_maker, qtys, 0.0))
cum_pv_ask  = np.cumsum(np.where(~is_buyer_maker, pv,   0.0))
cum_qty_ask = np.cumsum(np.where(~is_buyer_maker, qtys, 0.0))

def rolling_vwap(times_ns, cum_pv, cum_qty, window_s):
    window_ns  = int(window_s * 1e9)
    lefts      = np.searchsorted(times_ns, times_ns - window_ns, side="left")
    pv_window  = cum_pv  - np.where(lefts > 0, cum_pv[lefts - 1],  0.0)
    qty_window = cum_qty - np.where(lefts > 0, cum_qty[lefts - 1], 0.0)
    with np.errstate(invalid="ignore"):
        return pv_window / qty_window  # NaN when no trades on that side in window

df_agg_trades1 = df_agg_trades1.with_columns(
    [pl.Series(f"vwap_bid_{w}s", rolling_vwap(times_ns, cum_pv_bid, cum_qty_bid, w)) for w in [5, 15, 30]] +
    [pl.Series(f"vwap_ask_{w}s", rolling_vwap(times_ns, cum_pv_ask, cum_qty_ask, w)) for w in [5, 15, 30]]
)


In [ ]:
df_agg_trades1.head(15)

### Compute spread

In [ ]:
df_book_ticker1 = df_book_ticker1.with_columns(
    (pl.col("best_ask_price") - pl.col("best_bid_price")).alias("spread")
)

In [ ]:
# set max displayed rows by polars to 50
pl.Config.set_tbl_rows(100)

In [ ]:
df_book_ticker1.limit(15).collect()

In [ ]:
df_book_ticker1.select("spread").collect().describe()

In [ ]:
df_bt = df_book_ticker1.collect()
idx = df_bt["spread"].arg_max()

In [ ]:
df_bt[max(0, idx - 10) : idx + 90]

In [ ]:
t0 = datetime(2023, 7, 13, 21, 9, 45, 146000, tzinfo=timezone.utc)
t1 = t0 + timedelta(seconds=15)
df_liq_snap1.filter(pl.col("time_datetime").is_between(t0, t1))

In [ ]:
# Trailing realized volatility of mid_price log returns
# LOCF (last observation carried forward) resampling: for each sample point take 
# last observed price up to that time
df_bt_rv       = df_book_ticker1.select(["event_time_datetime", "mid_price"]).collect()
raw_times_ns   = df_bt_rv["event_time_datetime"].cast(pl.Int64).to_numpy()
raw_log_prices = np.log(df_bt_rv["mid_price"].to_numpy())

configs = [
    ("5s",   5,   0.1),   # 5s window,   100ms sampling
    ("30s",  30,  0.3),   # 30s window,  300ms sampling
    ("1min", 60,  0.5),   # 1min window, 500ms sampling
    ("5min", 300, 2.0),   # 5min window, 2s sampling
]

def compute_rvol(raw_times_ns, raw_log_prices, t_ns, window_s, freq_s):
    t0_ns   = t_ns - int(window_s * 1e9)
    freq_ns = int(freq_s * 1e9)
    # sample points from t0 up to and including t (LOCF — no lookahead)
    sample_times = np.arange(t0_ns, t_ns + 1, freq_ns)
    idxs  = np.searchsorted(raw_times_ns, sample_times, side="right") - 1
    valid = idxs >= 0
    if valid.sum() < 2:
        return float("nan")
    log_rets = np.diff(raw_log_prices[idxs[valid]])
    return float(np.sqrt(np.sum(log_rets ** 2)))

liq_times_ns = df_liq_snap1["time_datetime"].cast(pl.Int64).to_numpy()

for window_str, window_s, freq_s in configs:
    df_liq_snap1 = df_liq_snap1.with_columns(
        pl.Series(
            f"rvol_{window_str}",
            [compute_rvol(raw_times_ns, raw_log_prices, t, window_s, freq_s) for t in liq_times_ns],
        )
    )

df_liq_snap1.head(5)

In [ ]:
df_liq_snap1.filter(pl.col("agg_qty_5s_before_5s_after") == pl.col("agg_qty_5s_before_5s_after").max())

In [ ]:
df_mid_100ms = (
    df_book_ticker1
    .select(["event_time_datetime", "mid_price"])
    .collect()
    .upsample(time_column="event_time_datetime", every="100ms")
    .with_columns(pl.col("mid_price").forward_fill())
)
df_mid_100ms.head(10)

In [ ]:
df_book_ticker1.limit(30).collect()

In [ ]:
df_mid_100ms.head(5)

In [ ]:
log_prices = np.log(df_mid_100ms["mid_price"].to_numpy())
n          = len(log_prices)

def rvol_on_100ms_grid(log_prices, window_s, freq_s):
    stride    = int(round(freq_s / 0.1))           # steps between samples in 100ms grid
    n_samples = int(round(window_s / freq_s)) + 1  # sample points in window (inclusive)
    n_ret     = n_samples - 1                       # log returns per window

    sub     = log_prices[::stride]   # subsample to the desired freq
    sq_rets = np.diff(sub) ** 2      # squared log returns on subsampled series
    cum_sq  = np.cumsum(sq_rets)

    # rolling sum of n_ret consecutive squared returns via cumsum trick
    rv_sub = np.full(len(sub), np.nan)
    k      = np.arange(n_ret, len(sub))
    prev   = k - n_ret - 1
    rv_sub[n_ret:] = np.sqrt(
        cum_sq[k - 1] - np.where(prev >= 0, cum_sq[prev], 0.0)
    )

    # scatter back to full 100ms grid and forward-fill
    rv_full           = np.full(n, np.nan)
    rv_full[::stride] = rv_sub
    valid_mask        = ~np.isnan(rv_full)
    fill_idx          = np.where(valid_mask, np.arange(n), -1)
    np.maximum.accumulate(fill_idx, out=fill_idx)
    rv_full           = np.where(fill_idx >= 0, rv_full[np.maximum(fill_idx, 0)], np.nan)
    return rv_full

df_mid_100ms = df_mid_100ms.with_columns([
    pl.Series(f"rvol_{label}", rvol_on_100ms_grid(log_prices, window_s, freq_s))
    for label, window_s, freq_s in configs
])
df_mid_100ms = df_mid_100ms.drop_nans()


In [ ]:
start = 600
df_mid_100ms[start:start+20]

In [ ]:
t0 = datetime(2023, 6, 25, tzinfo=timezone.utc)
t1 = datetime(2023, 6, 27, tzinfo=timezone.utc)

df_vol_plot = df_mid_100ms.filter(
    (pl.col("event_time_datetime") >= t0) & (pl.col("event_time_datetime") < t1)
)
df_liq_plot = df_liq_snap1.filter(
    (pl.col("time_datetime") >= t0) & (pl.col("time_datetime") < t1)
).sort("time_datetime")

fig = go.Figure()

# Left y-axis: realized volatility from 100ms grid
for col, label, color in [
    ("rvol_5s",   "5s",   "steelblue"),
    ("rvol_30s",  "30s",  "darkorange"),
    ("rvol_1min", "1min", "seagreen"),
    ("rvol_5min", "5min", "mediumpurple"),
]:
    fig.add_trace(go.Scatter(
        x=df_vol_plot["event_time_datetime"].to_list(),
        y=df_vol_plot[col].to_list(),
        mode="lines",
        name=f"RVol {label}",
        line=dict(color=color, width=1),
        yaxis="y1",
    ))

for side, bar_color, line_color in [
    ("BUY",  "rgba(220, 50,  50,  0.5)", "rgba(220, 50,  50,  0.7)"),
    ("SELL", "rgba(50,  160, 50,  0.5)", "rgba(50,  160, 50,  0.7)"),
]:
    df_side    = df_liq_plot.filter(pl.col("side") == side)
    times_side = df_side["time_datetime"].to_list()

    # Bars at half the previous width
    fig.add_trace(go.Bar(
        x=times_side,
        y=df_side["original_quantity"].to_list(),
        name=f"Liq {side}",
        marker_color=bar_color,
        yaxis="y2",
        width=30_000,
    ))

    # Full-height vertical lines via hidden y3 spanning [0, 1]
    xs, ys = [], []
    for t in times_side:
        xs += [t, t, None]
        ys += [0,  1, None]
    fig.add_trace(go.Scatter(
        x=xs, y=ys,
        mode="lines",
        line=dict(color=line_color, width=1),
        yaxis="y3",
        showlegend=False,
    ))

fig.update_layout(
    title=f"Realized volatility (100ms grid) and liquidation sizes — first 2 days ({start_date})",
    xaxis=dict(title="Time (UTC)", rangeslider=dict(visible=True)),
    yaxis=dict(title="Realized volatility"),
    yaxis2=dict(title="Liquidation size (contracts)", overlaying="y", side="right"),
    yaxis3=dict(overlaying="y", range=[0, 1], visible=False),
    barmode="overlay",
    legend=dict(orientation="h", y=1.06),
    template="plotly_white",
    height=550,
)
fig.show()